# MAD-X style zero-shot Tamil emotion classification

English emotion **task adapter** (trained) + Tamil **language adapter** (swapped in at inference). No Tamil emotion labels are used for training, so you still need a labelled Tamil test set to measure real performance.

**Google Colab setup**
1. `Runtime -> Change runtime type -> T4 GPU` (training on CPU would take many hours).
2. Run the install cell. If Colab shows **Restart session**, click it and continue from the next cell (do not re-run the install cell).
3. Keep your labelled Tamil CSV (`text`, `label`) in Google Drive, or upload it when asked in the last section.

In [2]:
from datasets import load_dataset

# Load the dair-ai/emotion dataset
dataset = load_dataset('dair-ai/emotion')

# Display the dataset details and some samples
print(dataset)
print("\nTrain sample:")
display(dataset['train'][0])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/16000 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})

Train sample:


{'text': 'i didnt feel humiliated', 'label': 0}

In [1]:
# Colab already has torch, transformers, pandas and scikit-learn.
# `adapters` may change the transformers version. If Colab shows "Restart session", click it,
# then continue from the next cell (do NOT re-run this cell).
%pip install -q adapters datasets accelerate sentencepiece

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.5/295.5 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 104.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 123.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.


In [3]:
import os, random
import numpy as np
import torch

from datasets import load_dataset
from sklearn.metrics import accuracy_score, f1_score, classification_report
from transformers import (
    AutoTokenizer,
    TrainingArguments,
    DataCollatorWithPadding,
    set_seed,
)
from adapters import AutoAdapterModel, AdapterTrainer
from adapters.composition import Stack

import transformers, adapters
print("torch", torch.__version__, "| transformers", transformers.__version__, "| adapters", adapters.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU. Use Runtime -> Change runtime type -> T4 GPU, then rerun from the top.")

torch 2.11.0+cu130 | transformers 4.57.6 | adapters 1.3.0
CUDA available: True
GPU: Tesla T4


In [5]:
# ---------------- Configuration ----------------
BASE_MODEL = "FacebookAI/xlm-roberta-base"
EN_ADAPTER_ID = "AdapterHub/xlm-roberta-base-en-wiki_pfeiffer"
TA_ADAPTER_ID = "AdapterHub/xlm-roberta-base-ta-wiki_pfeiffer"
DATASET_ID = "dair-ai/emotion"
DATASET_CONFIG = "split"

try:
    from google.colab import drive
    print("Attempting to mount Google Drive...")
    drive.mount("/content/drive")
    OUTPUT_DIR = "/content/drive/MyDrive/madx_tamil_emotion"   # final adapter + tokenizer (persists in Drive)
except Exception as e:
    print(f"Google Drive mount failed: {e}\nFalling back to local storage instead.")
    OUTPUT_DIR = "/content/madx_tamil_emotion"                 # Fallback local directory

CKPT_DIR = "/content/checkpoints"                          # temporary checkpoints (fast local disk)
os.makedirs(OUTPUT_DIR, exist_ok=True)
TASK_ADAPTER_NAME = "emotion_task"
MAX_LENGTH = 128
NUM_EPOCHS = 4
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-4
SEED = 42

set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

Attempting to mount Google Drive...
Mounted at /content/drive


## 1. English emotion data

In [6]:
dataset = load_dataset(DATASET_ID, DATASET_CONFIG)
print(dataset)

# Take label names from the dataset itself so the order can never go out of sync.
LABEL_NAMES = dataset["train"].features["label"].names
ID2LABEL = {i: n for i, n in enumerate(LABEL_NAMES)}
LABEL2ID = {n: i for i, n in ID2LABEL.items()}
print("Labels:", LABEL_NAMES)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 2000
    })
})
Labels: ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise']


In [7]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

tokenized = dataset.map(tokenize_batch, batched=True, remove_columns=["text"])
tokenized = tokenized.rename_column("label", "labels")   # Trainer expects "labels"
collator = DataCollatorWithPadding(tokenizer=tokenizer)
tokenized

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/16000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 16000
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 2000
    })
})

## 2. Model, language adapters, task adapter

In [8]:
model = AutoAdapterModel.from_pretrained(BASE_MODEL)

# source="hf" loads the adapters from the Hugging Face Hub.
en_name = model.load_adapter(EN_ADAPTER_ID, load_as="english", source="hf", with_head=False)
ta_name = model.load_adapter(TA_ADAPTER_ID, load_as="tamil", source="hf", with_head=False)
print("Loaded:", en_name, ta_name)

# New task adapter + 6-class head
model.add_adapter(TASK_ADAPTER_NAME, config="pfeiffer")
model.add_classification_head(
    TASK_ADAPTER_NAME,
    num_labels=len(LABEL_NAMES),
    id2label=ID2LABEL,
)

# Freeze everything except the task adapter and head
model.train_adapter(TASK_ADAPTER_NAME)

# Training-time stack: English language adapter -> emotion task adapter
# (must be set AFTER train_adapter, which resets the active setup)
model.set_active_adapters(Stack("english", TASK_ADAPTER_NAME))
model.active_head = TASK_ADAPTER_NAME

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {trainable:,} / {total:,} ({100 * trainable / total:.3f}%)")
print("Active adapters:", model.active_adapters)

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Some weights of XLMRobertaAdapterModel were not initialized from the model checkpoint at FacebookAI/xlm-roberta-base and are newly initialized: ['heads.default.3.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

pytorch_adapter.bin:   0%|          | 0.00/29.6M [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

adapter_config.json: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

Loaded: english tamil
Trainable parameters: 2,331,864 / 295,151,064 (0.790%)
Active adapters: Stack[english, emotion_task]


## 3. Train the English task adapter

In [9]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "macro_f1": f1_score(labels, preds, average="macro"),
        "weighted_f1": f1_score(labels, preds, average="weighted"),
    }

training_args = TrainingArguments(
    output_dir=CKPT_DIR,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,
    remove_unused_columns=False,     # keep columns intact for the adapter model
    seed=SEED,
)

# AdapterTrainer (not Trainer) saves/loads only the adapter weights in checkpoints.
trainer = AdapterTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    data_collator=collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Macro F1,Weighted F1
1,0.365900,0.290902,0.906500,0.882855,0.907211
2,0.219300,0.246331,0.920500,0.894000,0.920717
3,0.173200,0.193535,0.931500,0.905285,0.931425
4,0.143700,0.185894,0.929500,0.904360,0.929527


TrainOutput(global_step=4000, training_loss=0.3298544223308563, metrics={'train_runtime': 411.2333, 'train_samples_per_second': 155.629, 'train_steps_per_second': 9.727, 'total_flos': 2017138448054016.0, 'train_loss': 0.3298544223308563, 'epoch': 4.0})

In [10]:
print("English validation:", trainer.evaluate(tokenized["validation"]))
print("English test:      ", trainer.evaluate(tokenized["test"]))

English validation: {'eval_loss': 0.19353508949279785, 'eval_accuracy': 0.9315, 'eval_macro_f1': 0.9052851725247791, 'eval_weighted_f1': 0.9314249063285372, 'eval_runtime': 2.707, 'eval_samples_per_second': 738.834, 'eval_steps_per_second': 23.273, 'epoch': 4.0}
English test:       {'eval_loss': 0.1975126564502716, 'eval_accuracy': 0.926, 'eval_macro_f1': 0.8826124217793918, 'eval_weighted_f1': 0.9253188512270215, 'eval_runtime': 2.7884, 'eval_samples_per_second': 717.257, 'eval_steps_per_second': 22.594, 'epoch': 4.0}


In [11]:
# Save the task adapter (with its head) and tokenizer
task_adapter_dir = os.path.join(OUTPUT_DIR, "emotion_task_adapter")
os.makedirs(task_adapter_dir, exist_ok=True)
model.save_adapter(task_adapter_dir, TASK_ADAPTER_NAME, with_head=True)
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, "tokenizer"))
print("Saved to:", task_adapter_dir)

Saved to: /content/drive/MyDrive/madx_tamil_emotion/emotion_task_adapter


## 4. Swap in the Tamil language adapter (zero-shot)

In [12]:
# Task adapter stays unchanged; only the language adapter changes.
model.set_active_adapters(Stack("tamil", TASK_ADAPTER_NAME))
model.active_head = TASK_ADAPTER_NAME
model.to(device)
model.eval()
print("Active adapters:", model.active_adapters)

def predict_proba(texts, allowed=None, batch_size=32):
    """Return an (N, num_labels) probability array.
    allowed: optional list of label names; all other classes are masked out."""
    mask = None
    if allowed is not None:
        mask = torch.full((len(LABEL_NAMES),), float("-inf"))
        for l in allowed:
            mask[LABEL2ID[l]] = 0.0
        mask = mask.to(device)

    out = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i + batch_size], return_tensors="pt", padding=True,
                        truncation=True, max_length=MAX_LENGTH).to(device)
        with torch.no_grad():
            logits = model(**enc).logits
        if mask is not None:
            logits = logits + mask
        out.append(torch.softmax(logits, dim=-1).cpu())
    return torch.cat(out).numpy()

def predict_emotion(text, allowed=None):
    p = predict_proba([text], allowed=allowed)[0]
    best = int(p.argmax())
    return {
        "text": text,
        "predicted_emotion": ID2LABEL[best],
        "confidence": float(p[best]),
        "all_scores": {ID2LABEL[i]: round(float(p[i]), 4) for i in range(len(LABEL_NAMES))},
    }

examples = [
    "இன்று எனக்கு மிகவும் மகிழ்ச்சியாக இருக்கிறது.",
    "அவன் செய்ததை நினைத்தாலே எனக்கு கோபமாக இருக்கிறது.",
    "என் நண்பரைப் பிரிந்ததால் மிகவும் வருத்தமாக இருக்கிறது.",
    "நாளைய முடிவைப் பற்றி எனக்குப் பயமாக இருக்கிறது.",
    "இந்தச் செய்தியை நம்பவே முடியவில்லை!",
]

for s in examples:
    r = predict_emotion(s)
    print(f"\n{r['text']}\n-> {r['predicted_emotion']} ({r['confidence']:.3f})  {r['all_scores']}")

Active adapters: Stack[tamil, emotion_task]

இன்று எனக்கு மிகவும் மகிழ்ச்சியாக இருக்கிறது.
-> joy (0.999)  {'sadness': 0.0005, 'joy': 0.9993, 'love': 0.0, 'anger': 0.0001, 'fear': 0.0, 'surprise': 0.0}

அவன் செய்ததை நினைத்தாலே எனக்கு கோபமாக இருக்கிறது.
-> anger (0.993)  {'sadness': 0.0016, 'joy': 0.0001, 'love': 0.0, 'anger': 0.9928, 'fear': 0.0053, 'surprise': 0.0}

என் நண்பரைப் பிரிந்ததால் மிகவும் வருத்தமாக இருக்கிறது.
-> sadness (0.947)  {'sadness': 0.9475, 'joy': 0.0011, 'love': 0.0001, 'anger': 0.0508, 'fear': 0.0005, 'surprise': 0.0}

நாளைய முடிவைப் பற்றி எனக்குப் பயமாக இருக்கிறது.
-> fear (0.925)  {'sadness': 0.0055, 'joy': 0.0003, 'love': 0.0, 'anger': 0.0686, 'fear': 0.9254, 'surprise': 0.0002}

இந்தச் செய்தியை நம்பவே முடியவில்லை!
-> fear (0.983)  {'sadness': 0.002, 'joy': 0.0036, 'love': 0.0, 'anger': 0.0031, 'fear': 0.9834, 'surprise': 0.0079}


## 5. Evaluate on a labelled Tamil test set

Expected CSV columns: `text`, `label`. Labels must be lowercase names from the English label set (`sadness, joy, love, anger, fear, surprise`).

If your Tamil data uses fewer classes (e.g. joy / sadness / anger / fear), pass them in `ALLOWED` so the English-only classes (`love`, `surprise`) cannot be predicted.

In [13]:
import pandas as pd

TAMIL_CSV = "/content/drive/MyDrive/tamil_test.csv"   # <- path to your CSV in Drive
if not os.path.exists(TAMIL_CSV):
    from google.colab import files
    print("CSV not found in Drive. Upload it now (columns: text,label), or cancel to skip:")
    uploaded = files.upload()
    TAMIL_CSV = next(iter(uploaded), None)
ALLOWED = None                         # e.g. ["joy", "sadness", "anger", "fear"]

def evaluate_tamil_test(texts, gold_labels, allowed=None):
    unknown = set(gold_labels) - set(LABEL2ID)
    if unknown:
        raise ValueError(f"Unknown labels {unknown}; expected a subset of {LABEL_NAMES}")

    probs = predict_proba(list(texts), allowed=allowed)
    y_pred = probs.argmax(axis=1)
    y_true = np.array([LABEL2ID[g] for g in gold_labels])

    present = sorted(set(y_true) | set(y_pred))
    print("Accuracy :", round(accuracy_score(y_true, y_pred), 4))
    print("Macro-F1 :", round(f1_score(y_true, y_pred, average="macro", zero_division=0), 4))
    print(classification_report(
        y_true, y_pred, labels=present,
        target_names=[ID2LABEL[i] for i in present], zero_division=0))
    return y_true, y_pred

if TAMIL_CSV and os.path.exists(TAMIL_CSV):
    df = pd.read_csv(TAMIL_CSV)
    df["label"] = df["label"].astype(str).str.strip().str.lower()
    y_true, y_pred = evaluate_tamil_test(df["text"].astype(str).tolist(), df["label"].tolist(), allowed=ALLOWED)
else:
    print("No Tamil test file provided - add one to run this cell.")

CSV not found in Drive. Upload it now (columns: text,label), or cancel to skip:


Saving Tamil_Emotion_tweets (2).xlsx to Tamil_Emotion_tweets (2).xlsx


UnicodeDecodeError: 'utf-8' codec can't decode bytes in position 15-16: invalid continuation byte